# DAF-03: Delhi Station and Sensor Discovery

We are checking which Delhi locations report PM2.5 and what other sensors exist at each location. We will collect every location and every sensor, then create a separate PM2.5 table for the station-selection decision.

## 1. Load the project configuration

The API key stays in `.env` instead of inside this notebook. This keeps the secret out of the code and out of Git. We only print whether the key exists, never the key itself.

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

project_root = Path.cwd().parent
load_dotenv(project_root / ".env")

api_key = os.getenv("OPENAQ_API_KEY")
if not api_key:
    raise RuntimeError("OPENAQ_API_KEY is missing from the project .env file")

headers = {"X-API-Key": api_key}
locations_url = "https://api.openaq.org/v3/locations"

print("API key loaded:", bool(api_key))

API key loaded: True


## 2. Create one reusable API request helper

We will call OpenAQ many times. Keeping the HTTP and error handling in one function prevents us from repeating the same checks in every request.

- `401` means the key is missing or invalid, so we stop.
- `429` means too many requests, so we wait and retry.
- `200` means the request succeeded.

In [2]:
def request_json(url, params=None, max_retries=3):
    for attempt in range(max_retries):
        response = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=30,
        )

        if response.status_code == 200:
            return response.json()

        if response.status_code == 401:
            raise RuntimeError("Authentication failed: check the OpenAQ API key")

        if response.status_code == 429:
            wait_seconds = 10 * (attempt + 1)
            print(f"Rate limit reached. Waiting {wait_seconds} seconds.")
            time.sleep(wait_seconds)
            continue

        raise RuntimeError(
            f"OpenAQ request failed with HTTP {response.status_code}: "
            f"{response.text[:200]}"
        )

    raise RuntimeError("OpenAQ request failed after several retries")

## 3. Fetch all Delhi locations

The API returns at most 100 locations per page. The first request returned 100 and reported more than 100, so we must continue with page 2 and any later pages. Stopping after page 1 would silently lose locations.

In [3]:
all_delhi_locations = []
page = 1
page_size = 100

while True:
    page_params = {
        "bbox": "76.80,28.40,77.40,28.90",
        "parameters_id": 2,
        "limit": page_size,
        "page": page,
    }

    page_payload = request_json(locations_url, params=page_params)
    page_locations = page_payload["results"]
    all_delhi_locations.extend(page_locations)

    print(f"Page {page}: {len(page_locations)} locations")

    if len(page_locations) < page_size:
        break

    page += 1
    time.sleep(1)

print("Total locations collected:", len(all_delhi_locations))

Page 1: 100 locations
Page 2: 2 locations
Total locations collected: 102


## 4. Fetch every sensor at every location

A location is a monitoring station. A station can have several sensors: PM2.5, NO2, ozone, and others. We keep every sensor so the table shows the complete picture, including first and last reading times.

In [4]:
all_sensor_records = []

for index, location in enumerate(all_delhi_locations, start=1):
    location_id = location["id"]
    sensors_url = f"https://api.openaq.org/v3/locations/{location_id}/sensors"
    sensors_payload = request_json(sensors_url)
    sensors = sensors_payload["results"]

    coordinates = location.get("coordinates") or {}
    provider = location.get("provider")
    if isinstance(provider, dict):
        provider = provider.get("name")

    for sensor in sensors:
        parameter = sensor.get("parameter") or {}
        first_reading = sensor.get("datetimeFirst") or {}
        last_reading = sensor.get("datetimeLast") or {}

        all_sensor_records.append(
            {
                "location_id": location_id,
                "sensor_id": sensor.get("id"),
                "location_name": location.get("name"),
                "parameter": parameter.get("name"),
                "latitude": coordinates.get("latitude"),
                "longitude": coordinates.get("longitude"),
                "provider": provider,
                "first_reading": first_reading.get("local"),
                "last_reading": last_reading.get("local"),
            }
        )

    if index % 10 == 0 or index == len(all_delhi_locations):
        print(
            f"Processed {index}/{len(all_delhi_locations)} locations; "
            f"sensor rows collected: {len(all_sensor_records)}"
        )

    time.sleep(1)

print("Total sensor records:", len(all_sensor_records))

Processed 10/102 locations; sensor rows collected: 99
Rate limit reached. Waiting 10 seconds.
Processed 20/102 locations; sensor rows collected: 175
Processed 30/102 locations; sensor rows collected: 305
Processed 40/102 locations; sensor rows collected: 377
Rate limit reached. Waiting 10 seconds.
Rate limit reached. Waiting 20 seconds.
Processed 50/102 locations; sensor rows collected: 441
Processed 60/102 locations; sensor rows collected: 621
Processed 70/102 locations; sensor rows collected: 764
Processed 80/102 locations; sensor rows collected: 914
Processed 90/102 locations; sensor rows collected: 1036
Processed 100/102 locations; sensor rows collected: 1130
Processed 102/102 locations; sensor rows collected: 1154
Total sensor records: 1154


## 5. Display all locations and sensors

A DataFrame gives us a readable table instead of only progress messages. The first table contains every sensor. The second table contains only PM2.5 rows for the station coverage decision.

In [5]:
all_sensors_table = pd.DataFrame(all_sensor_records)

print("Rows in sensor table:", len(all_sensors_table))
print("Locations in sensor table:", all_sensors_table["location_id"].nunique())

display(
    all_sensors_table
    .sort_values(["location_id", "parameter"])
    .reset_index(drop=True)
)

pm25_table = all_sensors_table[
    all_sensors_table["parameter"].fillna("").str.lower() == "pm25"
].copy()

print("PM2.5 rows:", len(pm25_table))
display(pm25_table.sort_values("location_id").reset_index(drop=True))

Rows in sensor table: 1154
Locations in sensor table: 102


,location_id,sensor_id,location_name,parameter,latitude,longitude,provider,first_reading,last_reading
0,13,13866,"Delhi Technological University, Delhi - CPCB",no2,28.744000,77.120000,CPCB,2016-11-03T00:30:00+05:30,2018-02-22T09:30:00+05:30
1,13,24,"Delhi Technological University, Delhi - CPCB",o3,28.744000,77.120000,CPCB,None,None
2,13,13864,"Delhi Technological University, Delhi - CPCB",pm25,28.744000,77.120000,CPCB,2016-11-03T00:30:00+05:30,2018-02-22T09:30:00+05:30
3,15,27,IGI Airport,co,28.560000,77.094000,CPCB,None,None
4,15,28,IGI Airport,no2,28.560000,77.094000,CPCB,None,None
...,...,...,...,...,...,...,...,...,...
1149,6299678,15890342,"Ved Vihar-Loni, Ghaziabad - UPPCB",relativehumidity,28.739153,77.273668,N/A,2026-04-07T22:30:00+05:30,2026-09-11T16:00:00+05:30
1150,6299678,15890343,"Ved Vihar-Loni, Ghaziabad - UPPCB",so2,28.739153,77.273668,N/A,2026-04-07T22:30:00+05:30,2026-09-11T16:00:00+05:30
1151,6299678,15890344,"Ved Vihar-Loni, Ghaziabad - UPPCB",temperature,28.739153,77.273668,N/A,2026-04-07T22:30:00+05:30,2026-09-11T16:00:00+05:30
1152,6299678,15890345,"Ved Vihar-Loni, Ghaziabad - UPPCB",wind_direction,28.739153,77.273668,N/A,2026-04-07T22:30:00+05:30,2026-09-11T16:00:00+05:30


PM2.5 rows: 147


,location_id,sensor_id,location_name,parameter,latitude,longitude,provider,first_reading,last_reading
0,13,13864,"Delhi Technological University, Delhi - CPCB",pm25,28.744000,77.120000,CPCB,2016-11-03T00:30:00+05:30,2018-02-22T09:30:00+05:30
1,15,30,IGI Airport,pm25,28.560000,77.094000,CPCB,None,None
2,16,34,Civil Lines,pm25,28.678700,77.226200,CPCB,None,None
3,17,12234787,"R K Puram, Delhi - DPCC",pm25,28.563262,77.186937,CPCB,2025-02-19T01:45:00+05:30,2026-09-11T15:45:00+05:30
4,17,35,"R K Puram, Delhi - DPCC",pm25,28.563262,77.186937,CPCB,2016-02-05T20:25:00+05:30,2018-02-22T02:45:00+05:30
...,...,...,...,...,...,...,...,...,...
142,6254665,15554728,"Commonwealth Sports Complex, Delhi - DPCC",pm25,28.615828,77.271992,N/A,2026-02-27T22:30:00+05:30,2026-09-11T16:00:00+05:30
143,6254666,15554740,"IGNOU_Maidan Garhi, Delhi - DPCC",pm25,28.493624,77.201159,N/A,2026-02-27T22:30:00+05:30,2026-09-11T16:00:00+05:30
144,6257818,15578291,"Cantonment Area, Delhi - DPCC",pm25,28.594169,77.125100,N/A,2026-03-02T18:30:00+05:30,2026-09-11T14:30:00+05:30
145,6299494,15888918,"Prashant Garden, Khora - UPPCB",pm25,28.611190,77.342060,N/A,2026-04-07T18:15:00+05:30,2026-09-11T16:00:00+05:30
